# 1. Load Data

In [1]:
import pandas as pd

def load_data(data_path : str):
    df = pd.read_csv(f"{data_path}")

    return df

def save_data(df: pd.DataFrame, data_path : str):

    return df.to_csv(f"{data_path}", index = False)

installments_payments = load_data("D:/4. Home Credit/Dataset/CSV/installments_payments.csv")

In [2]:
installments_payments_copy = installments_payments.copy()
installments_payments_copy.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


# 2. Feature Engineering and Aggregation

### Feature Engineering

In [3]:
# Days past due
installments_payments_copy['DPD'] = (installments_payments_copy['DAYS_ENTRY_PAYMENT'] - installments_payments_copy['DAYS_INSTALMENT']).clip(lower = 0)
installments_payments_copy['IS_LATE'] = (installments_payments_copy['DPD'] > 0).astype(int)

# Days before due
installments_payments_copy['DBD'] = (installments_payments_copy['DAYS_INSTALMENT'] - installments_payments_copy['DAYS_ENTRY_PAYMENT']).clip(lower = 0)

# Payment
installments_payments_copy['PAYMENT_DIFF'] = installments_payments_copy['AMT_INSTALMENT'] - installments_payments_copy['AMT_PAYMENT']
installments_payments_copy['PAYMENT_RATIO'] = installments_payments_copy['AMT_PAYMENT'] / (installments_payments_copy['AMT_INSTALMENT'] + 1e-9)
installments_payments_copy['IS_UNDERPAID'] = (installments_payments_copy['PAYMENT_DIFF'] > 0).astype(int)

# L1M, L3M, L6M
recent_futures = []

for month in [12,24]:
    recent = installments_payments_copy[installments_payments_copy['DAYS_INSTALMENT'].between(-30 * month, 0)]
    agg = recent.groupby("SK_ID_CURR").agg(**{
        f"INSTALMENTS_PAYMENT_AMT_PAYMENT_L{month}M_MEAN" : ("AMT_PAYMENT", "mean"),
        f"INSTALMENTS_PAYMENT_AMT_PAYMENT_L{month}M_SUM" : ("AMT_PAYMENT", "sum"),
        f"INSTALMENTS_PAYMENT_RECORD_PAYMENT_L{month}M_COUNT" : ("NUM_INSTALMENT_NUMBER", "size"),
        f"INSTALMENTS_PAYMENT_DPD_L{month}M_MEAN" : ("DPD", "mean"),
        f"INSTALMENTS_PAYMENT_DPD_L{month}M_MAX" : ("DPD", "max")
    })
    recent_futures.append(agg)

recent_futures = pd.concat(recent_futures, axis = 1).reset_index()

### Aggregation

In [4]:
# Drop unecessary column
installments_payments_copy = installments_payments_copy.drop(columns = ['SK_ID_PREV'])

# Aggregation dict
installments_payments_copy_aggregation = {}

installments_payments_copy_aggregation = {
    'NUM_INSTALMENT_VERSION': 'nunique',
    'NUM_INSTALMENT_NUMBER' : ['max'],
    'DAYS_INSTALMENT' : ['sum', 'mean'],
    'DAYS_ENTRY_PAYMENT' : ['sum', 'mean'],
    'AMT_INSTALMENT' : ['sum', 'min','max','mean'],
    'AMT_PAYMENT' : ['sum', 'min','max','mean'],
    'DPD' : ['sum', 'mean', 'max', 'min'],
    'DBD' : ['sum', 'mean', 'max', 'min']
}

# Aggregation
installments_payments_agg = installments_payments_copy.groupby("SK_ID_CURR").agg({**installments_payments_copy_aggregation})
installments_payments_agg.columns = [f"INSTALMENTS_{col}_{status}".upper() for col, status in installments_payments_agg.columns]
installments_payments_agg = installments_payments_agg.reset_index()
installments_payments_agg = installments_payments_agg.merge(recent_futures, on = 'SK_ID_CURR', how = 'left')
installments_payments_agg.head()

,SK_ID_CURR,INSTALMENTS_NUM_INSTALMENT_VERSION_NUNIQUE,INSTALMENTS_NUM_INSTALMENT_NUMBER_MAX,INSTALMENTS_DAYS_INSTALMENT_SUM,INSTALMENTS_DAYS_INSTALMENT_MEAN,INSTALMENTS_DAYS_ENTRY_PAYMENT_SUM,INSTALMENTS_DAYS_ENTRY_PAYMENT_MEAN,INSTALMENTS_AMT_INSTALMENT_SUM,INSTALMENTS_AMT_INSTALMENT_MIN,INSTALMENTS_AMT_INSTALMENT_MAX,...,INSTALMENTS_PAYMENT_AMT_PAYMENT_L12M_MEAN,INSTALMENTS_PAYMENT_AMT_PAYMENT_L12M_SUM,INSTALMENTS_PAYMENT_RECORD_PAYMENT_L12M_COUNT,INSTALMENTS_PAYMENT_DPD_L12M_MEAN,INSTALMENTS_PAYMENT_DPD_L12M_MAX,INSTALMENTS_PAYMENT_AMT_PAYMENT_L24M_MEAN,INSTALMENTS_PAYMENT_AMT_PAYMENT_L24M_SUM,INSTALMENTS_PAYMENT_RECORD_PAYMENT_L24M_COUNT,INSTALMENTS_PAYMENT_DPD_L24M_MEAN,INSTALMENTS_PAYMENT_DPD_L24M_MAX
0,100001,2,4,-15314.0,-2187.714286,-15365.0,-2195.000000,41195.925,3951.000,17397.900,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100002,2,19,-5605.0,-295.000000,-5993.0,-315.421053,219625.695,9251.775,53093.745,...,12905.2725,154863.27,12.0,0.0,0.0,11559.247105,219625.695,19.0,0.000000,0.0
2,100003,2,12,-34454.0,-1378.160000,-34633.0,-1385.320000,1618864.650,6662.970,560835.360,...,NaN,NaN,NaN,NaN,NaN,134438.827500,1344388.275,10.0,0.000000,0.0
3,100004,2,3,-2262.0,-754.000000,-2285.0,-761.666667,21288.465,5357.250,10573.965,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,2,9,-5274.0,-586.000000,-5486.0,-609.555556,56161.845,4813.200,17656.245,...,NaN,NaN,NaN,NaN,NaN,6240.205000,56161.845,9.0,0.111111,1.0


# 3. Save Data

In [5]:
save_data(installments_payments_agg, "D:/4. Home Credit/Dataset/CSV/instalments_payment_agg_final.csv")